## 1. Introduction to Supervised Learning: Regression
In a regression problem, we predict a continuous-valued output $y$ given input features $x$.
* **Hypothesis**: $h_\theta(x) = \sum_{i=0}^{d} \theta_i x_i = \theta^T x$
* **Parameters**: $\theta \in \mathbb{R}^{d+1}$
* **Cost Function**: Least-Squares (MSE)
$$J(\theta) = \frac{1}{2} \sum_{i=1}^{n} (h_\theta(x^{(i)}) - y^{(i)})^2$$

In [2]:
import numpy as np

def compute_cost(X, y, theta):
    """
    Computes the Mean Squared Error cost J(theta).
    X: (n x d+1) matrix, y: (n x 1) vector, theta: (d+1 x 1) vector
    """
    error = X @ theta - y
    return (1 / 2) * np.sum(error**2)

## 2. Optimization: Gradient Descent

### Batch Gradient Descent (BGD)
Update rule for all $j$:
$$\theta_j := \theta_j - \alpha \frac{\partial}{\partial \theta_j} J(\theta)$$
Vectorized form: $\theta := \theta - \alpha X^T (X\theta - y)$

In [3]:
def batch_gradient_descent(X, y, theta, alpha, iterations):
    n = len(y)
    for _ in range(iterations):
        # Vectorized gradient calculation
        gradient = X.T @ (X @ theta - y)
        theta = theta - alpha * gradient 
    return theta

### Stochastic Gradient Descent (SGD)
Update parameters based on a single training example $i$ at each step. Better for large-scale datasets where $n$ is massive.

In [4]:
def stochastic_gradient_descent(X, y, theta, alpha, iterations):
    n = len(y)
    for _ in range(iterations):
        for i in range(n):
            xi = X[i:i+1]
            yi = y[i:i+1]
            gradient = xi.T @ (xi @ theta - yi)
            theta = theta - alpha * gradient
    return theta

## 3. The Normal Equations (Analytical Solution)
Setting $\nabla_\theta J(\theta) = 0$ gives the closed-form solution:
$$\theta = (X^T X)^{-1} X^T y$$

**Note**: Inverting $X^T X$ takes $O(d^3)$, where $d$ is the number of features.

In [5]:
def normal_equations(X, y):
    """Analytical solution. Complexity: O(d^3) due to matrix inversion."""
    return np.linalg.inv(X.T @ X) @ X.T @ y

## 4. Probabilistic Interpretation
Assume $y^{(i)} = \theta^T x^{(i)} + \epsilon^{(i)}$ where $\epsilon^{(i)} \sim \mathcal{N}(0, \sigma^2)$.
Maximizing the Log-Likelihood:
$$\ell(\theta) = \log \prod_{i=1}^{n} p(y^{(i)} | x^{(i)}; \theta)$$
Leads directly to minimizing the sum of squares. This justifies the choice of MSE for Gaussian noise.

## 5. Locally Weighted Linear Regression (LWR)
A **non-parametric** algorithm. To predict at $x$, solve:
$$\min_\theta \sum_{i} w^{(i)} (y^{(i)} - \theta^T x^{(i)})^2$$
Where $w^{(i)} = \exp\left(-\frac{\|x^{(i)} - x\|^2}{2\tau^2}\right)$.

In [6]:
def lwr_predict(X_train, y_train, x_query, tau):
    # Compute weights relative to query point
    weights = np.exp(-np.sum((X_train - x_query)**2, axis=1) / (2 * tau**2))
    W = np.diag(weights)
    
    # Weighted Normal Equation
    theta = np.linalg.pinv(X_train.T @ W @ X_train) @ X_train.T @ W @ y_train
    return x_query @ theta